# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant JSON-LD schema, accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print some key metadata
md = dataset.metadata
print(f"Dataset name: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Published: {md.datePublished if hasattr(md, 'datePublished') else '[unknown]'}\n")
print(f"Identifier: {md.identifier if hasattr(md, 'identifier') else '[unknown]'}\n")
print(f"License: {md.license if hasattr(md, 'license') else '[unknown]'}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list all record sets, their `@id` values, and for each record set, its `field` `@id`s and (if present) any `column` `@id`s.


In [ ]:
# List all record sets with their @ids and field/column info
def get_id(obj):
    """Safely get @id if present."""
    if hasattr(obj, "id"):
        return obj.id
    if hasattr(obj, "@id"):
        return obj["@id"]
    if hasattr(obj, "_id"):
        return obj._id
    if hasattr(obj, '__getitem__') and '@id' in obj:
        return obj['@id']
    return str(obj)

# Get all record set objects
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in schema.")
else:
    for i, record_set in enumerate(record_sets):
        print(f"[{i}] Record Set name: {getattr(record_set, 'name', '[no name]')} @id: {get_id(record_set)}")

        fields = getattr(record_set, "fields", [])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"   - {getattr(field, 'name', '[no name]')} @id: {get_id(field)}")
                columns = getattr(field, "columns", None)
                if columns:
                    print("      Columns:")
                    for col in columns:
                        print(f"         * {getattr(col, 'name', '[no name]')} @id: {get_id(col)}")
        else:
            print("  No fields found.")
        print()


## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis. Each record set and its fields are referenced by their `@id`.

In [ ]:
# Extract data from all record sets as DataFrames, indexed by their @id
df_dict = {}
record_set_ids = []
for rs in record_sets:
    rs_id = get_id(rs)
    record_set_ids.append(rs_id)
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df_dict[rs_id] = pd.DataFrame(records)
        else:
            print(f"No records available for record set {rs_id}")
    except Exception as e:
        print(f"Skipping record set {rs_id}: error: {e}")

# Show available DataFrames and preview their columns
for rs_id, df in df_dict.items():
    print(f"Record set @id: {rs_id}")
    print(f"Columns: {list(df.columns)}")
    display(df.head(3))
    print()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records by a numeric field, normalize values, and group by a key attribute. All field names are referenced by their `@id` as found above.

Pick one record set with records, and select suitable field `@id`s for demonstration below.

In [ ]:
# Example: Pick first DataFrame with numeric fields for demonstration
import numpy as np

# Select one data frame
demo_df = None
demo_rsid = None

for rs_id, df in df_dict.items():
    # Try to find numeric field
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        demo_df = df
        demo_rsid = rs_id
        break
if demo_df is None:
    print("No record set with numeric fields found.")
else:
    print(f"Using record set @id: {demo_rsid}")
    print(f"Numeric fields detected: {numeric_fields}")
    numeric_field_id = numeric_fields[0]
    # Filter for numeric_field > mean
    threshold = demo_df[numeric_field_id].mean()
    filtered_df = demo_df[demo_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a non-numeric field (if exists)
    group_candidates = [col for col in demo_df.columns if col != numeric_field_id and demo_df[col].dtype == object]
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions for selected fields from the chosen record set.

In [ ]:
import matplotlib.pyplot as plt

if demo_df is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    plt.hist(demo_df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id exists, show boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(9,4))
        demo_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and analyze a FAIR-compliant dataset using the `mlcroissant` library. All schema entities (record sets, fields, columns) are referenced using their `@id`. This structured approach supports reproducible and robust ML data workflows on open science datasets.

> **Next steps:** Use the available DataFrames for statistical analysis, modeling, or integration with your ML pipeline!